# EEG_08 — TemporalChebGCN per Imagined Speech Decoding

## Architettura
Modello originale che combina:
1. **Temporal Encoder** (1D CNN per-nodo): comprime la serie temporale di ogni elettrodo in un embedding compatto
2. **ChebConv (K=2)**: convoluzione grafica su grafo di connettività funzionale
3. **Global mean pooling** + MLP classificatore

## Scelte di Design dalla Letteratura
Questo notebook prende **due scelte specifiche** da Lun et al. (2022):

> **Lun X, Jia Z, Hou Y, et al.**
> *GCNs-Net: A Graph Convolutional Neural Network Approach for Decoding Time-Resolved EEG Motor Imagery Signals*
> IEEE Transactions on Neural Systems and Rehabilitation Engineering, 2022, 30: 2447–2455.
> arXiv: [2006.08924](https://arxiv.org/abs/2006.08924)

**Scelta 1 — Grafo funzionale via PCC**: il grafo degli elettrodi è costruito calcolando la
matrice di correlazione di Pearson (|PCC|) tra canali sui dati di training, con soglia binaria.
Questo cattura la **connettività funzionale** tra elettrodi, non solo la prossimità anatomica.

**Scelta 2 — ChebConv (K=2)**: i filtri di Chebyshev al secondo ordine sono il tipo di
convoluzione grafica usato in GCNs-Net, empiricamente ottimale per EEG su grafo di elettrodi.

**Differenza rispetto a GCNs-Net**: GCNs-Net processa ogni timestep come campione indipendente
(approccio *time-resolved*, adatto a motor imagery). Qui usiamo invece un **Temporal Encoder**
(1D CNN) che processa il trial intero (384 campioni, ~1.5s), necessario per imagined speech
dove il contenuto semantico è distribuito sull'intera epoca.

## Pipeline
```
Input: grafo G per ogni trial
  Nodi V = 59 elettrodi EEG
  Archi E = coppie (i,j) con |PCC_ij| > soglia  [calcolato su dati training]
  Feature nodo = serie temporale raw (384 campioni)

[Temporal Encoder]  Conv1d(1→32, k=25) → Conv1d(32→64, k=10) → AvgPool → Linear(256→64)
[ChebConv 1]        ChebConv(64 → 128, K=2) + BN + ELU + Dropout
[ChebConv 2]        ChebConv(128 → 128, K=2) + BN + ELU
[Global MeanPool]   (batch, 128)
[MLP]               Linear(128→64) → ELU → Linear(64→n_classes)
```

## Setup
- **Split**: subject-independent — train sogg. 00-49, val 50-59, test 60-73 (identico a EEG_05)
- **Classi**: schema `ward4` (4 categorie semantiche) — chance level 25%
- **Env**: `daniele_311` (PyTorch Geometric 2.7.0)


In [ ]:
# ═══════════════════════════════════════════════════════════
#  TOGGLE CLASSI TARGET — modifica qui per cambiare schema
# ═══════════════════════════════════════════════════════════
USE_CLUSTERS   = True        # False → 110 parole originali
CLUSTER_SCHEME = "ward4"     # "ward4" | "ward5" | "sem5" | "pos4" | "concr4"
#                              (ignorato se USE_CLUSTERS = False)
# ───────────────────────────────────────────────────────────
PCC_THRESHOLD  = 0.5         # soglia su |PCC|: connetti coppie canali sopra soglia
CHEB_K         = 2           # ordine polinomi Chebyshev (GCNs-Net: K=2 ottimale)
NODE_EMB_DIM   = 64          # embedding per nodo (dopo temporal encoder)
GCN_HIDDEN     = 128         # canali nascosti nei layer ChebConv
# ═══════════════════════════════════════════════════════════


In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # fix OpenMP su macOS

import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

# PyTorch Geometric
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import ChebConv, global_mean_pool

# Device: MPS (Apple Silicon) > CUDA > CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:", device)
print("Python:", __import__('sys').version.split()[0])
print("torch:", torch.__version__)
import torch_geometric; print("torch_geometric:", torch_geometric.__version__)

In [ ]:
# ============================================================
# CONFIGURAZIONE
# ============================================================

project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents)
     if (p / ".git").exists()),
    Path().resolve()
)

META_CSV  = project_root / "data" / "interim" / "eeg_metadata.csv"
ELOC_PATH = project_root / "src" / "io" / "ebneuro.locs"
GEO_CSV   = project_root / "data" / "interim" / "geodesic_D_59_channels.csv"
interim_dir = project_root / "data" / "interim"

# Parametri EEG
N_CHANS  = 59    # canali dopo rimozione A1, A2
N_TIMES  = 384   # campioni a 256 Hz (~1.5s)
SFREQ    = 256

# Training
BATCH_SIZE   = 32
MAX_EPOCHS   = 100
PATIENCE     = 15
LR           = 1e-3
WEIGHT_DECAY = 1e-4

# Split subject-independent (identico a EEG_05)
SUBJ_TRAIN = list(range(50))
SUBJ_VAL   = list(range(50, 60))
SUBJ_TEST  = list(range(60, 74))

# Resume
RESUME = True

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("Config OK")
print(f"project_root: {project_root}")

In [ ]:
# ============================================================
# METADATA + CLUSTER MAPPING
# ============================================================

import sys
sys.path.insert(0, str(project_root / "scripts"))
from utils import load_label_scheme

meta = pd.read_csv(META_CSV)

# Filtro righe corrotte (epoche fuori range note)
_initial_len = len(meta)
meta = meta[~(
    (meta["path_h5"].str.contains("08_05.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_01.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_03.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_04.h5") & (meta["epoch_idx"] >= 34))
)]
if len(meta) < _initial_len:
    print(f"Rimosse {_initial_len - len(meta)} righe corrotte.")
meta["subject_id"] = meta["subject_id"].astype(str).str.zfill(2)

# Indici canali: rimuovi A1, A2
def read_eloc_names(path):
    names = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                names.append(parts[3])
    return names[:61]

ch_names_61 = read_eloc_names(ELOC_PATH)
EXCLUDE  = {"A1", "A2"}
keep_idx = [i for i, n in enumerate(ch_names_61) if n not in EXCLUDE]
keep_names = [ch_names_61[i] for i in keep_idx]
assert len(keep_idx) == N_CHANS, f"Attesi {N_CHANS} canali, trovati {len(keep_idx)}"

# Carica schema etichettatura via toggle
_scheme = CLUSTER_SCHEME if USE_CLUSTERS else "raw110"
labelid2cluster, N_CLASSES, cluster_names = load_label_scheme(_scheme, interim_dir)

RESULTS_CSV = project_root / "data" / "interim" / f"eeg08_gcn_{N_CLASSES}_results.csv"

print(f"Schema: {'clustering ' + CLUSTER_SCHEME if USE_CLUSTERS else '110 parole'}")
print(f"Classi: {N_CLASSES}  |  Chance level: {100/N_CLASSES:.1f}%")
print(f"Canali: {len(keep_names)} — {keep_names[:5]}...")
print(f"Totale record: {len(meta)}")

In [ ]:
# ============================================================
# GRAFO FUNZIONALE: adiacenza PCC calcolata sui dati di training
# Basato su: Lun et al. 2022 (GCNs-Net) — A = |PCC| con soglia
# ============================================================

def compute_pcc_graph(meta_df, keep_idx, subj_train_ids, threshold=0.5, n_samples=1000, seed=42):
    """
    Costruisce matrice di adiacenza funzionale via Pearson Correlation Coefficient.

    Approccio da GCNs-Net (Lun et al., 2022, IEEE TNSRE):
      1. Calcola |PCC| tra ogni coppia di canali su n_samples trial di training
      2. Media su tutti i trial campionati
      3. Rimuove auto-loop (diagonale = 0)
      4. Applica soglia: mantieni arco se |PCC| > threshold

    Args:
        threshold: soglia su |PCC| (default 0.5 — valore comune in letteratura)
        n_samples: trial da campionare per stima robusta della PCC

    Returns:
        edge_index: (2, E) torch.long — grafo bidirezionale
        pcc_matrix: (59, 59) np.float32 — matrice PCC media (per visualizzazione)
    """
    rng = np.random.RandomState(seed)
    train_records = meta_df[meta_df["subject_id"].isin(subj_train_ids)][["path_h5", "epoch_idx"]]
    n = min(n_samples, len(train_records))
    sample_idx = rng.choice(len(train_records), n, replace=False)
    sampled = train_records.iloc[sample_idx]

    paths_map = defaultdict(list)
    for _, row in sampled.iterrows():
        paths_map[row["path_h5"]].append(int(row["epoch_idx"]))

    buf = []
    print(f"Calcolo PCC su {n} trial da {len(paths_map)} file H5...")
    for path, epoch_idxs in tqdm(paths_map.items(), desc="PCC H5", leave=False):
        with h5py.File(path, "r") as f:
            for e_idx in epoch_idxs:
                x = f["data"][e_idx][keep_idx, :].astype(np.float32)  # (59, T)
                buf.append(x)

    # Media di |PCC| su tutti i trial campionati
    pcc_sum = np.zeros((len(keep_idx), len(keep_idx)), dtype=np.float64)
    for x in buf:
        pcc_sum += np.abs(np.corrcoef(x))
    pcc_matrix = (pcc_sum / len(buf)).astype(np.float32)  # (59, 59)

    # Rimuovi auto-loop (diagonale = 0)
    np.fill_diagonal(pcc_matrix, 0.0)

    # Binarizza con soglia
    rows, cols = np.where(pcc_matrix > threshold)
    edge_index = torch.tensor([rows.tolist(), cols.tolist()], dtype=torch.long)

    n_nodes = len(keep_idx)
    n_edges = edge_index.shape[1]
    avg_degree = n_edges / n_nodes
    print(f"Grafo PCC (soglia={threshold}): {n_nodes} nodi, {n_edges} archi, grado medio {avg_degree:.1f}")
    if avg_degree < 2:
        print("⚠  Grafo molto sparso — considera di abbassare PCC_THRESHOLD")

    return edge_index, pcc_matrix


train_ids = [str(i).zfill(2) for i in SUBJ_TRAIN]
edge_index, pcc_matrix = compute_pcc_graph(
    meta, keep_idx, train_ids,
    threshold=PCC_THRESHOLD, n_samples=1000
)

# Visualizzazione
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(pcc_matrix, ax=axes[0], cmap="RdBu_r", vmin=0, vmax=1,
            xticklabels=False, yticklabels=False)
axes[0].set_title(f"Matrice |PCC| media ({len(train_ids)} soggetti training)")

adj_binary = (pcc_matrix > PCC_THRESHOLD).astype(float)
sns.heatmap(adj_binary, ax=axes[1], cmap="Blues",
            xticklabels=False, yticklabels=False)
axes[1].set_title(f"Adiacenza binarizzata (soglia={PCC_THRESHOLD})")
plt.suptitle("Grafo funzionale EEG — adiacenza PCC (ispirato a Lun et al. 2022)", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# DATASET PyG: ogni trial → Data(x, edge_index, y)
# ============================================================

class EEGGraphDataset(Dataset):
    """
    Dataset PyG: ogni trial EEG diventa un grafo con N_CHANS nodi.
    
    - Nodi = elettrodi EEG
    - Feature per nodo = serie temporale raw (N_TIMES campioni)
    - Archi = grafo k-NN condiviso (edge_index immutabile)
    - Label = cluster_id (da labelid2cluster)
    
    Normalizzazione per-canale calcolata sul training set (mean, std passati a val/test).
    """
    def __init__(self, records, keep_idx, labelid2cluster, edge_index, mean=None, std=None):
        self.records = records
        self.keep_idx = keep_idx
        self.labelid2cluster = labelid2cluster
        self.edge_index = edge_index  # (2, E) — condiviso tra tutti i grafi
        self.mean = mean  # (59, 1) float32
        self.std  = std
        self.file_cache = {}
        if mean is None:
            self._compute_stats()

    def _compute_stats(self, seed=42):
        rng  = np.random.RandomState(seed)
        n    = min(500, len(self.records))
        idxs = rng.choice(len(self.records), n, replace=False)
        paths_map = defaultdict(list)
        for idx in idxs:
            r = self.records[idx]
            paths_map[r["path_h5"]].append(int(r["epoch_idx"]))
        buf = []
        print(f"Calcolo stats su {n} campioni da {len(paths_map)} file...")
        for path, epoch_idxs in tqdm(paths_map.items(), desc="Stats H5", leave=False):
            with h5py.File(path, "r") as f:
                for e_idx in epoch_idxs:
                    x = f["data"][e_idx][self.keep_idx, :].astype(np.float32)
                    buf.append(x)
        buf = np.stack(buf)  # (N, 59, T)
        self.mean = torch.tensor(
            buf.mean(axis=(0, 2), keepdims=True).squeeze(0), dtype=torch.float32
        )  # (59, 1)
        self.std  = torch.tensor(
            buf.std(axis=(0, 2), keepdims=True).squeeze(0).clip(1e-6), dtype=torch.float32
        )
        print(f"Stats OK | mean range: [{self.mean.min():.3f}, {self.mean.max():.3f}]")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        path = r["path_h5"]
        if path not in self.file_cache:
            self.file_cache[path] = h5py.File(path, "r")
        x = self.file_cache[path]["data"][int(r["epoch_idx"])][self.keep_idx, :].astype(np.float32)
        x = torch.tensor(x, dtype=torch.float32)
        x = (x - self.mean) / self.std  # normalizzazione per-canale
        label = self.labelid2cluster[int(r["label_idx"])]
        return Data(
            x          = x,                                            # (59, 384)
            edge_index = self.edge_index,                              # (2, E)
            y          = torch.tensor(label, dtype=torch.long),
        )

    def __del__(self):
        for f in self.file_cache.values():
            try: f.close()
            except: pass


def make_graph_splits(meta_df, labelid2cluster, subj_train, subj_val, subj_test, edge_index):
    """
    Crea train/val/test EEGGraphDataset con split subject-independent.
    
    Le stats di normalizzazione vengono calcolate SOLO sul training set
    e propagate a val e test.
    """
    def to_records(df_sub):
        return df_sub[["path_h5", "epoch_idx", "label_idx"]].to_dict("records")

    train_ids = [str(i).zfill(2) for i in subj_train]
    val_ids   = [str(i).zfill(2) for i in subj_val]
    test_ids  = [str(i).zfill(2) for i in subj_test]

    r_tr = to_records(meta_df[meta_df["subject_id"].isin(train_ids)])
    r_va = to_records(meta_df[meta_df["subject_id"].isin(val_ids)])
    r_te = to_records(meta_df[meta_df["subject_id"].isin(test_ids)])

    print(f"Split: train={len(r_tr)} | val={len(r_va)} | test={len(r_te)}")

    ds_tr = EEGGraphDataset(r_tr, keep_idx, labelid2cluster, edge_index)
    ds_va = EEGGraphDataset(r_va, keep_idx, labelid2cluster, edge_index, mean=ds_tr.mean, std=ds_tr.std)
    ds_te = EEGGraphDataset(r_te, keep_idx, labelid2cluster, edge_index, mean=ds_tr.mean, std=ds_tr.std)
    return ds_tr, ds_va, ds_te


print("EEGGraphDataset OK")

In [ ]:
# ============================================================
# MODELLI — TemporalChebGCN e varianti
# Grafo PCC + ChebConv(K=2): scelte da Lun et al. 2022 (GCNs-Net)
# Temporal Encoder: contributo originale (trial interi, imagined speech)
# ============================================================

class TemporalEncoder(nn.Module):
    """
    1D CNN per-nodo (pesi condivisi tra tutti gli elettrodi).
    Estrae embedding compatto dalla serie temporale di ogni canale EEG.

    Input:  (N_nodes_total, N_TIMES)
    Output: (N_nodes_total, out_dim)
    """
    def __init__(self, n_times: int = N_TIMES, out_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=25, stride=2, padding=12),  # T/2 = 192
            nn.BatchNorm1d(32),
            nn.ELU(),
            nn.Conv1d(32, 64, kernel_size=10, stride=2, padding=5),  # T/4 = 96
            nn.BatchNorm1d(64),
            nn.ELU(),
            nn.AdaptiveAvgPool1d(4),
            nn.Flatten(),
            nn.Linear(256, out_dim),
            nn.ELU(),
        )

    def forward(self, x):
        return self.net(x.unsqueeze(1))  # (N_nodes_total, out_dim)


class TemporalChebGCN(nn.Module):
    """
    TemporalChebGCN per imagined speech decoding (59 canali, trial interi).

    Design:
    - Temporal Encoder (1D CNN per-nodo): estrae embedding compatto da 384 timestep
    - ChebConv(K=2): convoluzione grafica spettrale su grafo PCC — da Lun et al. 2022
    - global_mean_pool + MLP: classificatore graph-level

    Nota: GCNs-Net (Lun et al., 2022) usa processing time-resolved (ogni timestep
    come campione separato). Il Temporal Encoder qui è una scelta originale,
    necessaria perché classifichiamo trial interi, non singoli timestep.

    Riferimento grafo+conv: Lun X et al., IEEE TNSRE 2022, arXiv:2006.08924
    """
    def __init__(self, n_times=N_TIMES, node_emb=NODE_EMB_DIM, gcn_hidden=GCN_HIDDEN,
                 n_classes=4, dropout=0.5, cheb_k=CHEB_K):
        super().__init__()
        self.temporal = TemporalEncoder(n_times, node_emb)
        self.conv1    = ChebConv(node_emb,   gcn_hidden, K=cheb_k)
        self.conv2    = ChebConv(gcn_hidden, gcn_hidden, K=cheb_k)
        self.drop     = nn.Dropout(dropout)
        self.bn1      = nn.BatchNorm1d(gcn_hidden)
        self.bn2      = nn.BatchNorm1d(gcn_hidden)
        self.classifier = nn.Sequential(
            nn.Linear(gcn_hidden, 64),
            nn.ELU(),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.temporal(x)                            # (N, node_emb)
        x = F.elu(self.bn1(self.conv1(x, edge_index)))  # ChebConv 1
        x = self.drop(x)
        x = F.elu(self.bn2(self.conv2(x, edge_index)))  # ChebConv 2
        x = global_mean_pool(x, batch)                  # (batch, gcn_hidden)
        return self.classifier(x)


class TemporalChebGCN3L(nn.Module):
    # Variante con 3 layer ChebConv (K=2)
    def __init__(self, n_times=N_TIMES, node_emb=NODE_EMB_DIM, gcn_hidden=GCN_HIDDEN,
                 n_classes=4, dropout=0.5, cheb_k=CHEB_K):
        super().__init__()
        self.temporal = TemporalEncoder(n_times, node_emb)
        self.conv1 = ChebConv(node_emb,   gcn_hidden, K=cheb_k)
        self.conv2 = ChebConv(gcn_hidden, gcn_hidden, K=cheb_k)
        self.conv3 = ChebConv(gcn_hidden, gcn_hidden, K=cheb_k)
        self.drop  = nn.Dropout(dropout)
        self.bn1   = nn.BatchNorm1d(gcn_hidden)
        self.bn2   = nn.BatchNorm1d(gcn_hidden)
        self.bn3   = nn.BatchNorm1d(gcn_hidden)
        self.classifier = nn.Sequential(
            nn.Linear(gcn_hidden, 64), nn.ELU(), nn.Dropout(0.3),
            nn.Linear(64, n_classes),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.temporal(x)
        x = F.elu(self.bn1(self.conv1(x, edge_index))); x = self.drop(x)
        x = F.elu(self.bn2(self.conv2(x, edge_index))); x = self.drop(x)
        x = F.elu(self.bn3(self.conv3(x, edge_index)))
        x = global_mean_pool(x, batch)
        return self.classifier(x)


class TemporalChebGCNSkip(nn.Module):
    # Variante con skip connection residuale tra layer ChebConv
    def __init__(self, n_times=N_TIMES, node_emb=NODE_EMB_DIM, gcn_hidden=GCN_HIDDEN,
                 n_classes=4, dropout=0.5, cheb_k=CHEB_K):
        super().__init__()
        self.temporal = TemporalEncoder(n_times, node_emb)
        self.proj  = nn.Linear(node_emb, gcn_hidden)
        self.conv1 = ChebConv(node_emb,   gcn_hidden, K=cheb_k)
        self.conv2 = ChebConv(gcn_hidden, gcn_hidden, K=cheb_k)
        self.drop  = nn.Dropout(dropout)
        self.bn1   = nn.BatchNorm1d(gcn_hidden)
        self.bn2   = nn.BatchNorm1d(gcn_hidden)
        self.classifier = nn.Sequential(
            nn.Linear(gcn_hidden, 64), nn.ELU(), nn.Dropout(0.3),
            nn.Linear(64, n_classes),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h    = self.temporal(x)
        skip = self.proj(h)
        h = F.elu(self.bn1(self.conv1(h, edge_index)))
        h = self.drop(h)
        h = F.elu(self.bn2(self.conv2(h, edge_index) + skip))
        h = global_mean_pool(h, batch)
        return self.classifier(h)


# Factory
GCN_MODELS = {
    "ChebGCN_2L":  lambda nc: TemporalChebGCN(n_classes=nc),
    "ChebGCN_3L":  lambda nc: TemporalChebGCN3L(n_classes=nc),
    "ChebGCNSkip": lambda nc: TemporalChebGCNSkip(n_classes=nc),
}

print(f"{'Modello':<22} {'Parametri':>12}")
print("-" * 36)
for name, factory in GCN_MODELS.items():
    m = factory(4)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"{name:<22} {n_params:>12,}")


In [ ]:
# ============================================================
# TRAINING + VALUTAZIONE (PyG)
# ============================================================

def train_gcn(model, ds_train, ds_val, save_path, tb_dir,
              n_epochs=MAX_EPOCHS, patience=PATIENCE,
              lr=LR, weight_decay=WEIGHT_DECAY, batch_size=BATCH_SIZE):
    """
    Training con early stopping su val_bacc e checkpoint.
    Restituisce (model, history_dict, n_epochs_trained).
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.CrossEntropyLoss()
    writer    = SummaryWriter(log_dir=str(tb_dir))

    loader_tr = PyGDataLoader(ds_train, batch_size=batch_size, shuffle=True)
    loader_va = PyGDataLoader(ds_val,   batch_size=batch_size, shuffle=False)

    best_val_bacc = -1.0
    best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    patience_cnt  = 0
    history       = defaultdict(list)

    for epoch in range(n_epochs):
        # ── Fase di training ──
        model.train()
        loss_sum, correct, n_tot = 0.0, 0, 0
        pbar = tqdm(loader_tr, desc=f"Epoch {epoch+1}/{n_epochs}", leave=False)
        for data in pbar:
            data = data.to(device)
            optimizer.zero_grad()
            logits = model(data)
            loss   = criterion(logits, data.y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            loss_sum += loss.item() * len(data.y)
            correct  += (logits.argmax(1) == data.y).sum().item()
            n_tot    += len(data.y)
            pbar.set_postfix(loss=f"{loss.item():.3f}")
        scheduler.step()

        # ── Fase di validazione ──
        model.eval()
        ys_v, ps_v = [], []
        with torch.no_grad():
            for data in loader_va:
                data = data.to(device)
                ps_v.extend(model(data).argmax(1).cpu().tolist())
                ys_v.extend(data.y.cpu().tolist())

        val_acc   = accuracy_score(ys_v, ps_v)
        val_bacc  = balanced_accuracy_score(ys_v, ps_v)
        train_acc = correct / n_tot if n_tot > 0 else 0.0
        train_loss_avg = loss_sum / n_tot if n_tot > 0 else 0.0

        history["val_acc"].append(val_acc)
        history["val_bacc"].append(val_bacc)
        history["train_acc"].append(train_acc)
        history["train_loss"].append(train_loss_avg)

        writer.add_scalar("val/acc",    val_acc,   epoch)
        writer.add_scalar("val/bacc",   val_bacc,  epoch)
        writer.add_scalar("train/acc",  train_acc, epoch)
        writer.add_scalar("train/loss", train_loss_avg, epoch)

        # ── Early stopping su val_bacc ──
        if val_bacc > best_val_bacc:
            best_val_bacc = val_bacc
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt  = 0
            save_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(model.state_dict(), save_path)
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f"    Early stop @ epoch {epoch+1} (best val_bacc={best_val_bacc:.3f})")
                break

    writer.close()
    model.load_state_dict(best_state)
    return model, dict(history), epoch + 1


def evaluate_gcn(model, ds, batch_size=64):
    """Calcola acc e bacc su un dataset PyG."""
    loader = PyGDataLoader(ds, batch_size=batch_size, shuffle=False)
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            ps.extend(model(data).argmax(1).cpu().tolist())
            ys.extend(data.y.cpu().tolist())
    return {
        "acc":  accuracy_score(ys, ps),
        "bacc": balanced_accuracy_score(ys, ps),
    }


print("Funzioni training OK")

In [ ]:
# ============================================================
# SANITY CHECK — forward pass su un mini-batch
# ============================================================

print("=== Sanity check forward pass ===")

# Crea dataset di test con i primi 10 record
records_sample = meta[["path_h5", "epoch_idx", "label_idx"]].iloc[:10].to_dict("records")
ds_sample = EEGGraphDataset(records_sample, keep_idx, labelid2cluster, edge_index)

sample_loader = PyGDataLoader(ds_sample, batch_size=4, shuffle=False)
batch_sample = next(iter(sample_loader))

print(f"Batch x shape:     {batch_sample.x.shape}     # (4×59, 384)")
print(f"Batch edge_index:  {batch_sample.edge_index.shape}")
print(f"Batch y:           {batch_sample.y}")
print(f"Batch batch tensor: {batch_sample.batch[:10]}...")

# Test su ogni modello
for mname, factory in GCN_MODELS.items():
    model_test = factory(N_CLASSES).to(device)
    model_test.eval()
    with torch.no_grad():
        out = model_test(batch_sample.to(device))
    print(f"{mname:<22} output: {out.shape}  ✓")

print("\nSanity check OK")

In [ ]:
# ============================================================
# ESPERIMENTO — GCN Subject-Independent
# ============================================================

chance_level = 1.0 / N_CLASSES
NORM_TAG     = "_norm" if USE_CLUSTERS else ""

TB_BASE   = project_root / "runs"   / f"eeg08_gcn_{N_CLASSES}"
CKPT_BASE = project_root / "models" / f"eeg08_gcn_{N_CLASSES}"
CKPT_BASE.mkdir(parents=True, exist_ok=True)

# ── Carica risultati già salvati (resume) ──
if RESUME and RESULTS_CSV.exists():
    df_existing = pd.read_csv(RESULTS_CSV)
    done_models = set(df_existing["model"].tolist())
    results_gcn = df_existing.to_dict("records")
    print(f"Resume: trovati {len(done_models)} modelli già completati: {done_models}")
else:
    done_models = set()
    results_gcn = []

# ── Split dataset ──
print("\nCreazione split subject-independent...")
ds_tr, ds_va, ds_te = make_graph_splits(
    meta, labelid2cluster, SUBJ_TRAIN, SUBJ_VAL, SUBJ_TEST, edge_index
)
print(f"Split OK | tr={len(ds_tr)} va={len(ds_va)} te={len(ds_te)}")

# ── Loop principale ──
print(f"\n── GCN Subject-Independent │ {N_CLASSES} classi │ k={K_NEIGHBORS} ──")
print(f"   Chance level: {chance_level:.1%}")
print()

for mname, factory in GCN_MODELS.items():
    if mname in done_models:
        print(f"  {mname:<22} ⏭ già completato")
        continue

    t0 = time.time()
    save_path = CKPT_BASE / f"{mname}.pth"
    tb_dir    = TB_BASE / mname

    model = factory(N_CLASSES)
    model, hist, n_ep = train_gcn(model, ds_tr, ds_va, save_path, tb_dir)

    va_r = evaluate_gcn(model, ds_va)
    te_r = evaluate_gcn(model, ds_te)
    elapsed = time.time() - t0

    row = dict(
        model    = mname,
        val_acc  = va_r["acc"],
        val_bacc = va_r["bacc"],
        test_acc = te_r["acc"],
        test_bacc= te_r["bacc"],
        epochs   = n_ep,
        time_s   = round(elapsed, 1),
        k_neighbors = K_NEIGHBORS,
        node_emb    = NODE_EMB_DIM,
        gcn_hidden  = GCN_HIDDEN,
        n_classes   = N_CLASSES,
    )
    results_gcn.append(row)
    done_models.add(mname)

    # Salva dopo ogni modello (resume-safe)
    pd.DataFrame(results_gcn).to_csv(RESULTS_CSV, index=False)

    print(f"  {mname:<22} val={va_r['acc']:.3f}  test={te_r['acc']:.3f}  "
          f"bacc={te_r['bacc']:.3f}  ({n_ep}ep {elapsed:.0f}s)")

print(f"\n✓ Completato  (chance={chance_level:.1%})")

In [ ]:
# ============================================================
# RISULTATI E VISUALIZZAZIONE
# ============================================================

df_gcn = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame(results_gcn)
chance = 1.0 / N_CLASSES

print(f"\n═══════════════════════════════════════════════")
print(f"  EEG_08 — GCN Spaziale | {N_CLASSES} classi | k={K_NEIGHBORS}")
print(f"  Chance level: {chance:.1%}")
print(f"═══════════════════════════════════════════════")
print(df_gcn[["model", "val_acc", "val_bacc", "test_acc", "test_bacc", "epochs"]].to_string(index=False))

best_row = df_gcn.sort_values("test_bacc", ascending=False).iloc[0]
print(f"\n► Miglior modello: {best_row['model']}")
print(f"  val_bacc={best_row['val_bacc']:.3f}  test_bacc={best_row['test_bacc']:.3f}")
print(f"  Ratio vs chance: {best_row['test_bacc']/chance:.2f}x")

In [ ]:
# ============================================================
# BAR CHART — confronto modelli GCN + EEGNet baseline
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"EEG_08 — GCN Spaziale ({N_CLASSES} classi, k={K_NEIGHBORS})", fontsize=14, fontweight="bold")

colors_model = ["#1565C0", "#2E7D32", "#6A1B9A"]
models_order = list(GCN_MODELS.keys())

# ── Accuracy panel ──
ax = axes[0]
x = np.arange(len(models_order))
df_plot = df_gcn.set_index("model").reindex(models_order)

bars_val  = ax.bar(x - 0.2, df_plot["val_acc"],  0.35, label="Val",  color="#1976D2", alpha=0.85)
bars_test = ax.bar(x + 0.2, df_plot["test_acc"], 0.35, label="Test", color="#43A047", alpha=0.85)
ax.axhline(chance, color="red", linestyle="--", linewidth=1.5, label=f"Chance ({chance:.1%})")
ax.set_xticks(x); ax.set_xticklabels([m.replace("Temporal", "T\n") for m in models_order], fontsize=10)
ax.set_ylabel("Accuracy"); ax.set_title("Accuracy (val / test)")
ax.legend(fontsize=9); ax.set_ylim(0, max(0.5, df_gcn["test_acc"].max() + 0.05))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))

# ── Balanced accuracy panel ──
ax = axes[1]
bars_val2  = ax.bar(x - 0.2, df_plot["val_bacc"],  0.35, label="Val",  color="#1976D2", alpha=0.85)
bars_test2 = ax.bar(x + 0.2, df_plot["test_bacc"], 0.35, label="Test", color="#43A047", alpha=0.85)
ax.axhline(chance, color="red", linestyle="--", linewidth=1.5, label=f"Chance ({chance:.1%})")
ax.set_xticks(x); ax.set_xticklabels([m.replace("Temporal", "T\n") for m in models_order], fontsize=10)
ax.set_ylabel("Balanced Accuracy"); ax.set_title("Balanced Accuracy (val / test)")
ax.legend(fontsize=9); ax.set_ylim(0, max(0.5, df_gcn["test_bacc"].max() + 0.05))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))

plt.tight_layout()

fig_path = project_root / "figures" / f"eeg08_gcn_{N_CLASSES}_results.png"
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
print(f"Salvato: {fig_path}")
plt.show()

In [ ]:
# ============================================================
# CONFUSION MATRIX — modello migliore su test set
# ============================================================

best_model_name = df_gcn.sort_values("test_bacc", ascending=False).iloc[0]["model"]
print(f"Confusion matrix per: {best_model_name}")

# Ricarica il modello best
best_model = GCN_MODELS[best_model_name](N_CLASSES)
ckpt_path  = CKPT_BASE / f"{best_model_name}.pth"
if ckpt_path.exists():
    best_model.load_state_dict(torch.load(ckpt_path, map_location="cpu"))
    best_model = best_model.to(device)
    
    loader_te = PyGDataLoader(ds_te, batch_size=64, shuffle=False)
    best_model.eval()
    ys_all, ps_all = [], []
    with torch.no_grad():
        for data in loader_te:
            data = data.to(device)
            ps_all.extend(best_model(data).argmax(1).cpu().tolist())
            ys_all.extend(data.y.cpu().tolist())
    
    cm = confusion_matrix(ys_all, ps_all)
    class_labels = [cluster_names.get(i, f"C{i}") for i in range(N_CLASSES)]
    
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=class_labels, yticklabels=class_labels, ax=ax
    )
    ax.set_xlabel("Predetto"); ax.set_ylabel("Reale")
    ax.set_title(f"Confusion Matrix — {best_model_name} (test set)")
    plt.tight_layout()
    plt.savefig(project_root / "figures" / f"eeg08_cm_{best_model_name}.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print(f"Checkpoint non trovato: {ckpt_path}")

In [ ]:
# ============================================================
# [OPZIONALE] HYPERPARAMETER TUNING — Optuna
# Esegui questa cella solo dopo aver verificato che il modello
# funziona e supera chance. Ottimizza su val_bacc.
# ============================================================
# pip install optuna (se non installato)

RUN_OPTUNA = False   # ← cambia a True per avviare il tuning
N_TRIALS   = 30      # numero di trial Optuna

if RUN_OPTUNA:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        # Spazio di ricerca degli iperparametri
        threshold  = trial.suggest_float("pcc_threshold", 0.3, 0.7, step=0.1)
        cheb_k     = trial.suggest_int("cheb_k", 1, 3)
        node_emb   = trial.suggest_categorical("node_emb_dim", [32, 64, 128])
        gcn_hidden = trial.suggest_categorical("gcn_hidden",   [64, 128, 256])
        dropout    = trial.suggest_float("dropout", 0.3, 0.6, step=0.1)

        # Ricostruisce grafo con la soglia PCC proposta
        ei, _ = compute_pcc_graph(
            meta, keep_idx, [str(i).zfill(2) for i in SUBJ_TRAIN],
            threshold=threshold, n_samples=500
        )
        if ei.shape[1] == 0:
            return 0.0  # grafo vuoto: scarta

        # Ricostruisce dataset con nuovo edge_index
        ds_tr_opt = EEGGraphDataset(
            meta[meta["subject_id"].isin([str(i).zfill(2) for i in SUBJ_TRAIN])][["path_h5","epoch_idx","label_idx"]].to_dict("records"),
            keep_idx, labelid2cluster, ei
        )
        ds_va_opt = EEGGraphDataset(
            meta[meta["subject_id"].isin([str(i).zfill(2) for i in SUBJ_VAL])][["path_h5","epoch_idx","label_idx"]].to_dict("records"),
            keep_idx, labelid2cluster, ei, mean=ds_tr_opt.mean, std=ds_tr_opt.std
        )

        model = TemporalChebGCN(
            node_emb=node_emb, gcn_hidden=gcn_hidden,
            n_classes=N_CLASSES, dropout=dropout, cheb_k=cheb_k
        )
        ckpt = CKPT_BASE / f"optuna_trial_{trial.number}.pth"
        tb   = TB_BASE / f"optuna_trial_{trial.number}"
        _, hist, _ = train_gcn(model, ds_tr_opt, ds_va_opt, ckpt, tb,
                               n_epochs=50, patience=10)  # epoche ridotte per velocità
        return max(hist["val_bacc"]) if hist["val_bacc"] else 0.0

    study = optuna.create_study(direction="maximize",
                                study_name=f"eeg08_gcn_{N_CLASSES}cls")
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    print(f"\nBest trial: {study.best_trial.number}")
    print(f"Best val_bacc: {study.best_value:.4f}")
    print(f"Best params:   {study.best_params}")

    # Visualizzazione importanza parametri
    try:
        fig = optuna.visualization.plot_param_importances(study)
        fig.show()
        fig2 = optuna.visualization.plot_optimization_history(study)
        fig2.show()
    except Exception:
        pass
else:
    print("Optuna disabilitato — imposta RUN_OPTUNA = True per avviare il tuning.")
    print("Consiglio: esegui prima il loop principale (cella precedente) e verifica")
    print("che almeno un modello superi chance prima di fare hyperparameter search.")
